In [1]:
import duckdb                                                                                                                                                                                                       
import numpy as np                                                                                                                                                                                                  
import pandas as pd                                                                                                                                                                                                 
from pathlib import Path                                                                                                                                                                                            
                                                                                                                                                                                                                      
WAREHOUSE = Path("/Users/premputtegowda/reps/reps-tracker/emerging_re/data/warehouse/emerging_re.duckdb")                                                                                                           
                                                                                                                                                                                                                      
  # Config knobs — mirror analog_search.py defaults                                                                                                                                                                   
POP_MIN, POP_MAX = 100_000, 300_000                                                                                                                                                                                 
DIVERSITY_HHI_MAX = 0.10                                                                                                                                                                                            
WINNER_TOP_PCT = 0.10       # top 10% per cohort year                                                                                                                                                               
TOP_K_ANALOGS = 5           # aggregate similarity: mean of top-K                                                                                                                                                   
HORIZON_YEARS = 3           # 36 months                                                                                                                                                                             
TEST_YEAR = 2022            # change to see other years                                                                                                                                                             
STATE_CAP = 2               # max picks per state in top-5                                                                                                                                                          
TOP_N = 5                                                                                                                                                                                                           
                                                                                                                                                                                                                      
pd.set_option("display.max_columns", None)                                                                                                                                                                          
pd.set_option("display.width", 200)       
                                                                                                                                                                                                                      
print(f"Warehouse exists: {WAREHOUSE.exists()}")

Warehouse exists: True


In [2]:
 con = duckdb.connect(str(WAREHOUSE), read_only=True)

In [3]:
 panel = con.execute(f"""                                                                                                                                                                                            
      SELECT f.*, mr.macro_regime                                                                                                                                                                                     
      FROM main_marts.mart_county_features f
      LEFT JOIN main_intermediate.int_national_macro_regime mr                                                                                                                                                        
          ON EXTRACT(YEAR FROM f.period_end_date) = mr.year
      WHERE EXTRACT(MONTH FROM f.period_end_date) = 12                                                                                                                                                                
        AND f.pop_latest BETWEEN {POP_MIN + 1} AND {POP_MAX}                                                                                                                                                          
        AND f.employment_diversity_hhi IS NOT NULL
        AND f.employment_diversity_hhi <= {DIVERSITY_HHI_MAX}                                                                                                                                                         
  """).df() 

In [6]:
panel["period_end_date"] = pd.to_datetime(panel["period_end_date"])                                                                                                                                                 
panel["year"] = panel["period_end_date"].dt.year

print(f"Panel shape: {panel.shape[0]:,} rows × {panel.shape[1]} columns")                                                                                                                                           
print(f"Year range: {panel.year.min()} - {panel.year.max()}")
print(f"\nCounts per year (last 10):")                                                                                                                                                                              
print(panel.groupby("year").size().tail(10).to_string())  

Panel shape: 8,538 rows × 102 columns
Year range: 2000 - 2025

Counts per year (last 10):
year
2016    331
2017    332
2018    330
2019    329
2020    325
2021    325
2022    324
2023    328
2024    326
2025    332


In [7]:
print(panel.groupby("year").size().tail(10).to_string())

year
2016    331
2017    332
2018    330
2019    329
2020    325
2021    325
2022    324
2023    328
2024    326
2025    332


In [9]:
#Cell 3
acs = con.execute("""                                                                                                                                                                                               
      SELECT fips, period_end_date, median_gross_rent                                                                                                                                                                 
      FROM main_staging.stg_census_acs                                                                                                                                                                                
      WHERE median_gross_rent IS NOT NULL                                                                                                                                                                             
      QUALIFY row_number() OVER (                                                                                                                                                                                     
          PARTITION BY fips, period_end_date ORDER BY as_of_date DESC
      ) = 1                                                                                                                                                                                                           
  """).df()
  acs["period_end_date"] = pd.to_datetime(acs["period_end_date"])                                                                                                                                                     
  acs["year"] = acs["period_end_date"].dt.year
                                                                                                                                                                                                                      
  # Forward-join trick: shift each row's year back by HORIZON_YEARS,
  # so joining on (fips, year) brings forward-year rent onto the current-year row.                                                                                                                                    
  forward = acs[["fips", "year", "median_gross_rent"]].rename(                                                                                                                                                        
      columns={"median_gross_rent": "median_gross_rent_forward"}
  )                                                                                                                                                                                                                   
  forward["year"] = forward["year"] - HORIZON_YEARS   # shift 2022 → 2019 so 2019 gets 2022's rent
                                                                                                                                                                                                                      
  acs_with_target = acs.merge(forward, on=["fips", "year"], how="left")
                                                                                                                                                                                                                      
  # Compute the raw forward return                                                                                                                                                                                    
  acs_with_target["acs_forward_return_36mo"] = (
      acs_with_target["median_gross_rent_forward"] / acs_with_target["median_gross_rent"] - 1                                                                                                                         
  )                                         

  # Demean by NATIONAL mean per cohort year — same convention as target_zori_*_demeaned in the mart.                                                                                                                  
  # Uses ALL counties in ACS (not just tertiary) so the demean reflects the true macro base rate.
  year_natl_mean = (                                                                                                                                                                                                  
      acs_with_target.dropna(subset=["acs_forward_return_36mo"])
      .groupby("year")["acs_forward_return_36mo"].mean()                                                                                                                                                              
  )                                                                                                                                                                                                                   
  acs_with_target["target_acs_rent_36mo_demeaned"] = (
      acs_with_target["acs_forward_return_36mo"] - acs_with_target["year"].map(year_natl_mean)                                                                                                                        
  )                                         

  print(f"ACS rows: {len(acs_with_target):,}")                                                                                                                                                                        
  print(f"Rows with observed 36mo forward target: {acs_with_target['target_acs_rent_36mo_demeaned'].notna().sum():,}")
  print(f"\nCoverage by year (with target vs without):")                                                                                                                                                              
  print(acs_with_target.groupby("year")["target_acs_rent_36mo_demeaned"].agg(['count', 'size']).tail(15).to_string())           

IndentationError: unexpected indent (3195321683.py, line 10)

In [19]:
acs = con.execute("""                                                                                                                                                                                               
      SELECT fips, period_end_date, median_gross_rent                                                                                                                                                                 
      FROM main_staging.stg_census_acs                                                                                                                                                                                
      WHERE median_gross_rent IS NOT NULL                                                                                                                                                                             
      QUALIFY row_number() OVER (                                                                                                                                                                                     
          PARTITION BY fips, period_end_date ORDER BY as_of_date DESC
      ) = 1                                                                                                                                                                                                           
  """).df()

In [20]:
acs["period_end_date"] = pd.to_datetime(acs["period_end_date"])                                                                                                                                                     
acs["year"] = acs["period_end_date"].dt.year

In [21]:
forward = acs[["fips", "year", "median_gross_rent"]].rename(                                                                                                                                                        
      columns={"median_gross_rent": "median_gross_rent_forward"}
  ) 

In [22]:
forward["year"] = forward["year"] - HORIZON_YEARS

In [23]:
acs_with_target = acs.merge(forward, on=["fips", "year"], how="left")

In [24]:
acs_with_target.head(10)

,fips,period_end_date,median_gross_rent,year,median_gross_rent_forward
0,12033,2011-12-31,869.0,2011,901.0
1,12053,2011-12-31,885.0,2011,904.0
2,12123,2011-12-31,569.0,2011,576.0
3,13205,2011-12-31,565.0,2011,579.0
4,18019,2011-12-31,739.0,2011,768.0
5,18099,2011-12-31,655.0,2011,667.0
6,19023,2011-12-31,562.0,2011,571.0
7,13001,2011-12-31,524.0,2011,514.0
8,13261,2011-12-31,608.0,2011,643.0
9,20197,2011-12-31,717.0,2011,685.0


In [25]:
 acs_with_target["acs_forward_return_36mo"] = (
      acs_with_target["median_gross_rent_forward"] / acs_with_target["median_gross_rent"] - 1                                                                                                                         
  )  

In [26]:
acs_with_target.head(10)

,fips,period_end_date,median_gross_rent,year,median_gross_rent_forward,acs_forward_return_36mo
0,12033,2011-12-31,869.0,2011,901.0,0.036824
1,12053,2011-12-31,885.0,2011,904.0,0.021469
2,12123,2011-12-31,569.0,2011,576.0,0.012302
3,13205,2011-12-31,565.0,2011,579.0,0.024779
4,18019,2011-12-31,739.0,2011,768.0,0.039242
5,18099,2011-12-31,655.0,2011,667.0,0.018321
6,19023,2011-12-31,562.0,2011,571.0,0.016014
7,13001,2011-12-31,524.0,2011,514.0,-0.019084
8,13261,2011-12-31,608.0,2011,643.0,0.057566
9,20197,2011-12-31,717.0,2011,685.0,-0.044630


In [28]:
 year_natl_mean = (                                                                                                                                                                                                  
      acs_with_target.dropna(subset=["acs_forward_return_36mo"])
      .groupby("year")["acs_forward_return_36mo"].mean()                                                                                                                                                              
  ) 

In [29]:
year_natl_mean

year
2008    0.093498
2009    0.074124
2010    0.052915
2011    0.049988
2012    0.056621
2013    0.080931
2014    0.085191
2015    0.074632
2016    0.093248
2017    0.159821
2018    0.182897
2019    0.164079
Name: acs_forward_return_36mo, dtype: float64

In [30]:
acs_with_target["target_acs_rent_36mo_demeaned"] = (
      acs_with_target["acs_forward_return_36mo"] - acs_with_target["year"].map(year_natl_mean)                                                                                                                        
  )  

In [32]:
acs_with_target.head()

,fips,period_end_date,median_gross_rent,year,median_gross_rent_forward,acs_forward_return_36mo,target_acs_rent_36mo_demeaned
0,12033,2011-12-31,869.0,2011,901.0,0.036824,-0.013164
1,12053,2011-12-31,885.0,2011,904.0,0.021469,-0.028519
2,12123,2011-12-31,569.0,2011,576.0,0.012302,-0.037686
3,13205,2011-12-31,565.0,2011,579.0,0.024779,-0.025209
4,18019,2011-12-31,739.0,2011,768.0,0.039242,-0.010746


In [34]:
print(f"ACS rows: {len(acs_with_target):,}")   

ACS rows: 48,252


In [35]:
print(f"Rows with observed 36mo forward target: {acs_with_target['target_acs_rent_36mo_demeaned'].notna().sum():,}")

Rows with observed 36mo forward target: 38,543


0         True
1         True
2         True
3         True
4         True
         ...  
48247    False
48248    False
48249    False
48250    False
48251    False
Name: target_acs_rent_36mo_demeaned, Length: 48252, dtype: bool

In [37]:
print(f"\nCoverage by year (with target vs without):")                                                                                                                                                              
print(acs_with_target.groupby("year")["target_acs_rent_36mo_demeaned"].agg(['count', 'size']).tail(15).to_string()) 


Coverage by year (with target vs without):
      count  size
year             
2008   3219  3219
2009   3220  3220
2010   3218  3220
2011   3218  3220
2012   3218  3220
2013   3216  3219
2014   3215  3219
2015   3209  3219
2016   3210  3217
2017   3200  3216
2018   3196  3211
2019   3204  3213
2020      0  3212
2021      0  3212
2022      0  3215


In [43]:
acs_with_target.groupby("year")["target_acs_rent_36mo_demeaned"].agg(['count','size'])

,count,size
year,,
2008,3219,3219
2009,3220,3220
2010,3218,3220
2011,3218,3220
2012,3218,3220
2013,3216,3219
2014,3215,3219
2015,3209,3219
2016,3210,3217


# High-Cap Rate Strategy — Top-5 per Cohort Year

Simple rule: **buy the 5 highest-cap markets** in the (population-band + employment-diverse) universe each year.

Metrics reported per pick:
- **Entry cap** = ACS `median_gross_rent × 12 / median_home_value` at entry date (cap_rate_proxy)
- **Exit cap** = same measure at entry + 3 years (or +2 years if 3yr not observable)
- **Cap Δ (bps)** = exit − entry (negative = compression = good for owner)
- **ZORI 36mo** = realized Zillow rent growth over horizon
- **Total return** = `(1 + rent_growth) × (cap_entry / cap_exit) − 1` — decomposes MF value appreciation into rent × cap compression

Universe filter always includes `employment_diversity_hhi ≤ 0.10` for structural diversification.

In [ ]:
# Reusable function — pull the high-cap picks + realized returns per year
import duckdb, pandas as pd, numpy as np

WAREHOUSE = "/Users/premputtegowda/reps/reps-tracker/emerging_re/data/warehouse/emerging_re.duckdb"

def high_cap_analysis(pop_min: int, pop_max: int, hhi_max: float = 0.10) -> pd.DataFrame:
    """Load panel + attach 3yr and 2yr exit caps + compute realized TR-ZORI36/24."""
    con = duckdb.connect(WAREHOUSE, read_only=True)
    df = con.execute(f'''
        SELECT f.fips, f.county_name, f.state_name, f.pop_latest,
               EXTRACT(YEAR FROM f.period_end_date)::INT AS year,
               f.cap_rate_proxy AS cap_entry,
               f.mf_cap_rate_estimated AS cap_entry_mf,
               f.zori_forward_return_36mo AS zori36,
               f.zori_forward_return_24mo AS zori24,
               f.employment_diversity_hhi AS hhi
        FROM main_marts.mart_county_features f
        WHERE EXTRACT(MONTH FROM f.period_end_date) = 12
          AND f.pop_latest > {pop_min}
          AND f.pop_latest <= {pop_max}
          AND f.employment_diversity_hhi IS NOT NULL
          AND f.employment_diversity_hhi <= {hhi_max}
    ''').df()
    con.close()

    exit3 = df[['fips','year','cap_entry']].rename(columns={'year':'ey3','cap_entry':'cap_exit_3yr'})
    df['ey3'] = df['year'] + 3
    df = df.merge(exit3, on=['fips','ey3'], how='left')
    exit2 = df[['fips','year','cap_entry']].rename(columns={'year':'ey2','cap_entry':'cap_exit_2yr'})
    df['ey2'] = df['year'] + 2
    df = df.merge(exit2, on=['fips','ey2'], how='left')

    df['tr36'] = (1 + df['zori36']) * (df['cap_entry'] / df['cap_exit_3yr']) - 1
    df['tr24'] = (1 + df['zori24']) * (df['cap_entry'] / df['cap_exit_2yr']) - 1
    return df


def print_top5_per_year(df: pd.DataFrame, min_year: int = 2015, max_year: int = 2025):
    """Print per-year top-5 by cap_entry with entry/exit cap, rent, and total return."""
    summary = []
    for yr in sorted(df['year'].unique()):
        if yr < min_year or yr > max_year:
            continue
        c = df[df['year'] == yr].dropna(subset=['cap_entry'])
        if len(c) < 20:
            continue
        top5 = c.nlargest(5, 'cap_entry')
        uni_tr = c['tr36'].mean(skipna=True) * 100
        top5_tr = top5['tr36'].mean(skipna=True) * 100
        edge = top5_tr - uni_tr if pd.notna(top5_tr) and pd.notna(uni_tr) else np.nan

        print(f"\n===== {yr} (n={len(c)}) | Top-5 TR: {top5_tr:+.2f}%  Universe TR: {uni_tr:+.2f}%  Edge: {edge:+.2f}pp =====")
        print(f"{'#':<3}{'Market':<40}{'Pop':>10}   {'Cap in':>8}  {'Cap out':>9}  {'Δ bps':>7}  {'ZORI36':>9}  {'TR':>10}  {'HHI':>7}")
        for i, (_, r) in enumerate(top5.iterrows(), 1):
            name = f"{r['county_name']}, {r['state_name']}"
            pop = f"{int(r['pop_latest']):,}"
            cap_e = f"{r['cap_entry']*100:.2f}%"
            if pd.notna(r['cap_exit_3yr']):
                cap_x = f"{r['cap_exit_3yr']*100:.2f}%"; d = f"{(r['cap_exit_3yr']-r['cap_entry'])*10000:+.0f}"
            elif pd.notna(r['cap_exit_2yr']):
                cap_x = f"{r['cap_exit_2yr']*100:.2f}%*"; d = f"{(r['cap_exit_2yr']-r['cap_entry'])*10000:+.0f}*"
            else:
                cap_x = "n/a"; d = "n/a"
            zori = f"{r['zori36']*100:+.2f}%" if pd.notna(r['zori36']) else (f"{r['zori24']*100:+.2f}%†" if pd.notna(r['zori24']) else "n/a")
            tr = f"{r['tr36']*100:+.2f}%" if pd.notna(r['tr36']) else (f"{r['tr24']*100:+.2f}%†" if pd.notna(r['tr24']) else "n/a")
            hhi = f"{r['hhi']:.4f}"
            print(f"{i:<3}{name:<40}{pop:>10}   {cap_e:>8}  {cap_x:>9}  {d:>7}  {zori:>9}  {tr:>10}  {hhi:>7}")
        summary.append({'year': yr, 'n': len(c), 'top5_TR': top5_tr, 'uni_TR': uni_tr, 'edge_pp': edge})
    print("\n* = 2yr exit used   † = 24mo return used\n")
    return pd.DataFrame(summary)

## Universe A — Mid-size metros (pop 300K-500K, diverse HHI ≤ 0.10)

~87-91 counties per year. Historical edge: **+7.4pp/cohort** on average (7 of 10 cohorts beat universe).

In [ ]:
df_300_500 = high_cap_analysis(pop_min=300_000, pop_max=500_000)
summary_300_500 = print_top5_per_year(df_300_500)

In [ ]:
print("=== SUMMARY: 300-500K + diverse ===")
print(summary_300_500.to_string(index=False, float_format=lambda x: f'{x:+.2f}'))
obs = summary_300_500.dropna(subset=['edge_pp'])
print(f"\nAvg edge: {obs['edge_pp'].mean():+.2f}pp   Won: {(obs['edge_pp']>0).sum()}/{len(obs)} cohorts")

## Universe B — Tertiary markets (pop 100K-300K, diverse HHI ≤ 0.10)

~325-332 counties per year. Historical edge: **+11.6pp/cohort** on average (8 of 8 cohorts beat universe).

Wider band = more inefficiency to exploit. Larger cap compressions (−200 to −400 bps common vs −50 to −150 in 300-500K).

In [ ]:
df_100_300 = high_cap_analysis(pop_min=100_000, pop_max=300_000)
summary_100_300 = print_top5_per_year(df_100_300)

In [ ]:
print("=== SUMMARY: 100-300K + diverse ===")
print(summary_100_300.to_string(index=False, float_format=lambda x: f'{x:+.2f}'))
obs = summary_100_300.dropna(subset=['edge_pp'])
print(f"\nAvg edge: {obs['edge_pp'].mean():+.2f}pp   Won: {(obs['edge_pp']>0).sum()}/{len(obs)} cohorts")

## Universe C — Combined view: which band offered a better edge?

In [ ]:
compare = pd.DataFrame({
    'year': summary_100_300['year'],
    '100-300K_top5': summary_100_300['top5_TR'].values,
    '300-500K_top5': summary_300_500.set_index('year').reindex(summary_100_300['year'])['top5_TR'].values,
    '100-300K_edge': summary_100_300['edge_pp'].values,
    '300-500K_edge': summary_300_500.set_index('year').reindex(summary_100_300['year'])['edge_pp'].values,
})
print("=== 100-300K vs 300-500K — high-cap top-5 side-by-side ===")
print(compare.to_string(index=False, float_format=lambda x: f'{x:+.2f}'))
print(f"\nAvg edge 100-300K: {compare['100-300K_edge'].mean():+.2f}pp")
print(f"Avg edge 300-500K: {compare['300-500K_edge'].mean():+.2f}pp")

## Returns by cap-rate quintile (100-300K universe)

Is the pattern monotonic — does higher cap → higher return across the whole distribution?

In [ ]:
obs = df_100_300.dropna(subset=['cap_entry','zori36','cap_exit_3yr']).copy()
obs = obs[(obs['year'] >= 2015) & (obs['year'] <= 2022)]

rows = []
for y in sorted(obs['year'].unique()):
    c = obs[obs['year'] == y].copy()
    c['quintile'] = pd.qcut(c['cap_entry'], q=5, labels=['Q1','Q2','Q3','Q4','Q5'], duplicates='drop')
    for q in ['Q1','Q2','Q3','Q4','Q5']:
        sub = c[c['quintile'] == q]
        if len(sub):
            rows.append({'year': y, 'quintile': q, 'n': len(sub),
                'avg_cap_pct': sub['cap_entry'].mean()*100,
                'avg_zori36_pct': sub['zori36'].mean()*100,
                'cap_delta_bps': (sub['cap_exit_3yr'].mean() - sub['cap_entry'].mean())*10000,
                'avg_TR_pct': sub['tr36'].mean()*100})
r = pd.DataFrame(rows)

print("=== TR-ZORI36 by cap quintile per year (%) ===")
print(r.pivot(index='year', columns='quintile', values='avg_TR_pct').round(2).to_string())

print("\n=== SUMMARY: avg per quintile across all cohorts ===")
print(r.groupby('quintile', observed=True).agg(
    avg_cap_pct=('avg_cap_pct','mean'),
    avg_cap_delta_bps=('cap_delta_bps','mean'),
    avg_zori36_pct=('avg_zori36_pct','mean'),
    avg_TR_pct=('avg_TR_pct','mean')).round(2).to_string())

## Today's picks — 2025 cohort (highest cap in each band)

These are the current recommendations from the high-cap rule. Cap Δ and TR not yet observable (need 3 years forward).

In [ ]:
for band_label, df in [('100-300K + diverse', df_100_300), ('300-500K + diverse', df_300_500)]:
    c25 = df[df['year'] == 2025].dropna(subset=['cap_entry']).nlargest(5, 'cap_entry')
    print(f"\n=== 2025 picks — {band_label} ===")
    for i, (_, r) in enumerate(c25.iterrows(), 1):
        print(f"  #{i}  {r['county_name']}, {r['state_name']:20s}  pop={int(r['pop_latest']):>7,}  "
              f"cap_ACS={r['cap_entry']*100:5.2f}%  cap_MF={r['cap_entry_mf']*100 if pd.notna(r['cap_entry_mf']) else float('nan'):5.2f}%  "
              f"HHI={r['hhi']:.4f}")